<a href="https://colab.research.google.com/github/jonathansearch/RATISS-ONE/blob/main/colab/Mini_Cerveau_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 MICRO-CERVEAU RATISS GPU T4 (PROTOTYPE ULTRA-PURIFIÉ)
**RATISS Labs · Auteur : Jonathan Evina (Mono-chercheur, Yaoundé, Cameroun) · Licence MIT**  
**Architecture Tenseur PyTorch GPU · 957 Concepts Clés Filtrés · 50 060 Synapses Actives · Inférence < 1 ms**

---
### 📋 Fiche Technique : Naissance du Micro-Cerveau RATISS GPU T4
> *Ce modèle a été conçu au laboratoire de recherche RATISS à Yaoundé (Cameroun) sous la supervision de Jonathan Evina pour valider le principe d'émergence sémantique par rapport au gros modèle RATISS-ONE.*

- **Vocabulaire** : 957 concepts clés uniques rigoureusement filtrés contre le bruit sémantique (stop-words et artefacts de prompt éliminés).
- **Architecture** : Matrice synaptique d'adjacence active gérée par PyTorch sur GPU T4 ( de taille ).
- **Entraînement** : Association Hebbienne auto-supervisée accélérée par GPU T4 sur 10 000 paires conversationnelles.
- **Vitesse de génération** : Moins de 1 ms par inférence grâce aux multiplications tensorielles GPU parallèles.
- **Format** : Fichier unifié  sérialisé pour la recherche cognitive.

### 📦 1/3 : Téléchargement du Mini-Cerveau et de sa Fiche Technique

In [ ]:
import os, urllib.request, time

print("📦 1/3 : Téléchargement du Mini-Cerveau GPU et de sa Notice...")

fichiers = {
    "mini_cerveau_gpu.pt": "https://raw.githubusercontent.com/jonathansearch/RATISS-ONE/main/colab/mini_cerveau_gpu.pt",
    "README_MINI_CERVEAU.md": "https://raw.githubusercontent.com/jonathansearch/RATISS-ONE/main/colab/README_MINI_CERVEAU.md"
}

for nom, url in fichiers.items():
    if not os.path.exists(nom):
        print(f"   📥 Récupération de {nom}...")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            with open(nom, "wb") as f:
                f.write(resp.read())
    print(f"   ✓ {nom} prêt ({os.path.getsize(nom)/(1024*1024):.2f} Mo)")

print("
📄 Contenu de la Fiche Technique officielle :
")
with open("README_MINI_CERVEAU.md", "r", encoding="utf-8") as f:
    print(f.read())

### ⚡ 2/3 : Chargement du Tenseur Synaptique PyTorch sur GPU (T4)

In [ ]:
import torch
import numpy as np

print("⚡ 2/3 : Chargement du Tenseur Synaptique sur GPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   • Dispositif d'exécution : {device.upper()}")

t0 = time.time()
modele = torch.load("mini_cerveau_gpu.pt", map_location=device)
dt_ms = (time.time() - t0) * 1000

vocab = modele["vocabulaire"]
idx = modele["index"]
W = modele["matrice_synaptique"]
meta = modele.get("metadata", {})

if not isinstance(W, torch.Tensor):
    W = torch.tensor(W, dtype=torch.float32, device=device)
else:
    W = W.to(device)

print("
" + "=" * 75)
print("🧠 AUDIT DU MICRO-CERVEAU GPU T4")
print("=" * 75)
print(f"⏱️ Temps de chargement : {dt_ms:.2f} ms")
print(f"🧬 Concepts uniques (V): {len(vocab):,} concepts filtrés sans bruit")
print(f"⚡ Synapses actives (E): {int((W > 0).sum().item()):,} connexions")
print(f"📐 Forme du tenseur   : {tuple(W.shape)} (Précision : {W.dtype})")
print(f"🎯 Densité synaptique  : {((W > 0).sum().item() / W.numel())*100:.2f}%")
print(f"🏛️ Laboratoire         : {meta.get('laboratoire', 'RATISS Labs, Yaoundé')}")
print("=" * 75)

### 💬 3/3 : Décodage Sémantique & Inférence Haute Vitesse (< 1 ms)

In [ ]:
import re

def interroger_mini_cerveau(message, top_k=2):
    """Inférence tensorielle pure sur GPU (< 1 ms)."""
    t_inf = time.time()
    tokens = [w.lower() for w in re.findall(r"\w+", message) if len(w) > 2]
    
    # Vecteur d'activation d'entrée h
    h = torch.zeros(len(vocab), dtype=torch.float32, device=device)
    mots_reconnus = []
    for t in tokens:
        if t in idx:
            h[idx[t]] += 2.0
            mots_reconnus.append(t)
            
    if len(mots_reconnus) == 0:
        return "Mes 957 concepts clés ne contiennent pas ces mots. Posez-moi une question générale !"
        
    # Projection synaptique tensorielle : s = W^T * h
    s = torch.matmul(W.t(), h)
    
    # Concept pivot émergent
    pivot_idx = torch.argmax(s).item()
    pivot_mot = vocab[pivot_idx]
    
    # Voisins immédiats du pivot dans le tenseur synaptique
    liens = W[pivot_idx]
    top_indices = torch.topk(liens, k=min(top_k + 3, len(vocab))).indices.tolist()
    
    mots_voisins = []
    for j in top_indices:
        if liens[j] > 0 and vocab[j] != pivot_mot and vocab[j] not in mots_voisins:
            mots_voisins.append(vocab[j])
            if len(mots_voisins) >= top_k:
                break
                
    dt_us = (time.time() - t_inf) * 1_000_000
    
    # Formulation en français limpide sans bruit
    if len(mots_voisins) >= 2:
        phrase = f"Sur le sujet de « {pivot_mot} », mes connexions s'associent directement avec « {mots_voisins[0]} » et « {mots_voisins[1]} »."
    elif len(mots_voisins) == 1:
        phrase = f"Le concept de « {pivot_mot} » est étroitement lié à « {mots_voisins[0]} »."
    else:
        phrase = f"Le concept central identifié dans mes circuits est « {pivot_mot} »."
        
    return f"{phrase} (Inférence GPU : {dt_us:.0f} µs)"

# Tests de démonstration immédiats
tests = [
    "Que peux-tu me dire sur la France ?",
    "Quelles sont les questions scientifiques ?",
    "Parle-moi de situation et de politique"
]

print("
🧪 DÉMONSTRATION DE PAROLE DU MINI-CERVEAU GPU :
")
for q in tests:
    print(f"👤 Question : « {q} »")
    print(f"🤖 Réponse  : {interroger_mini_cerveau(q)}
")